In [ ]:
# どこから実行されても chapter4（src があるフォルダ）をパスに追加
import sys
from pathlib import Path
def _find_chapter4():
    cwd = Path.cwd()
    for d in [cwd, cwd.parent, cwd / "chapter4", cwd.parent / "chapter4", cwd / "genai-agent-advanced-book" / "chapter4"]:
        if (d / "src" / "agent.py").exists():
            return d.resolve()
    return cwd
_chapter4 = _find_chapter4()
if str(_chapter4) not in sys.path:
    sys.path.insert(0, str(_chapter4))
print(f"使用中: {_chapter4}")

: 

In [4]:
import importlib

from src.agent import HelpDeskAgent
import src.configs as configs
from src.tools.search_xyz_manual import search_xyz_manual
from src.tools.search_xyz_qa import (
    search_xyz_qa,
)

# configs.py の変更が反映されるよう毎回リロード
importlib.reload(configs)
print(f"settings env_path: {configs._env_path} (exists={configs._env_path.is_file()})")

settings = configs.Settings()
print("Settings loaded")


ModuleNotFoundError: No module named 'src'

In [3]:
agent = HelpDeskAgent(
    settings=settings,
    tools=[search_xyz_manual, search_xyz_qa],
)

In [25]:
question = """
お世話になっております。

現在、XYZシステムの利用を検討しており、以下の2点についてご教示いただければと存じます。

1. パスワードに利用可能な文字の制限について
当該システムにてパスワードを設定する際、使用可能な文字の範囲（例：英数字、記号、文字数制限など）について詳しい情報をいただけますでしょうか。安全かつシステムでの認証エラーを防ぐため、具体的な仕様を確認したいと考えております。

2. 最新リリースの取得方法について
最新のアップデート情報をどのように確認・取得できるかについてもお教えいただけますと幸いです。

お忙しいところ恐縮ですが、ご対応のほどよろしくお願い申し上げます。
"""

# question = """
# お世話になっております。

# 現在、XYZシステムを利用を検討しており、以下の点についてご教示いただければと存じます。

# 1. 特定のプロジェクトに対してのみ通知を制限する方法について

# 2. パスワードに利用可能な文字の制限について
# 当該システムにてパスワードを設定する際、使用可能な文字の範囲（例：英数字、記号、文字数制限など）について詳しい情報をいただけますでしょうか。安全かつシステムでの認証エラーを防ぐため、具体的な仕様を確認したいと考えております。

# お忙しいところ恐縮ですが、ご対応のほどよろしくお願い申し上げます。

# """

## 計画ステップ

In [26]:
input_data = {"question": question}

plan_result = agent.create_plan(state=input_data)

In [ ]:
plan_result["plan"]

## ツール選択ステップ

In [28]:
input_data = {
    "question": question,
    "plan": plan_result["plan"],
    "subtask": plan_result["plan"][0],
    "challenge_count": 0,
    "is_completed": False,
}

In [29]:
select_tool_result = agent.select_tools(state=input_data)

In [ ]:
select_tool_result

In [31]:
select_tool_result["messages"][-1]

{'role': 'assistant',
 'tool_calls': [{'id': 'call_zM2g5tqao9s01G4uYwCtplrr',
   'function': {'arguments': '{"keywords":"パスワード 設定 文字制限"}',
    'name': 'search_xyz_manual'},
   'type': 'function'}]}

In [ ]:
select_tool_result["messages"]

## ツール実行ステップ

In [33]:
input_data = {
    "question": question,
    "plan": plan_result["plan"],
    "subtask": plan_result["plan"][0],
    "challenge_count": 0,
    "messages": select_tool_result["messages"],
    "is_completed": False,
}

In [ ]:
tool_results = agent.execute_tools(state=input_data)

In [ ]:
tool_results["tool_results"][0][0].results

In [ ]:
tool_results

## サブタスク回答

In [37]:
input_data = {
    "question": question,
    "plan": plan_result["plan"],
    "subtask": plan_result["plan"][0],
    "challenge_count": 0,
    "messages": tool_results["messages"],
    "tool_results": tool_results["tool_results"],
    "is_completed": False,
}

In [38]:
subtask_answer = agent.create_subtask_answer(state=input_data)

In [ ]:
subtask_answer

In [ ]:
print(subtask_answer["subtask_answer"])

## リフレクション

In [41]:
input_data = {
    "question": question,
    "plan": plan_result["plan"],
    "subtask": plan_result["plan"][0],
    "challenge_count": 0,
    "messages": subtask_answer["messages"],
    "tool_results": tool_results["tool_results"],
    "is_completed": False,
    "subtask_answer": subtask_answer["subtask_answer"],
}

In [42]:
reflection_result = agent.reflect_subtask(state=input_data)

In [ ]:
reflection_result

In [ ]:
# 最初に選択されたツールを確認
print(reflection_result["messages"][2]["tool_calls"][0]["function"]["name"])

In [ ]:
# リフレクション結果の確認
print("is_completed =", reflection_result["reflection_results"][0].is_completed)
print("advice =", reflection_result["reflection_results"][0].advice)